# Soft Actor-Critic (SAC)

Read TD3 first. Replace deterministic exploration noise with a learned stochastic policy and an entropy-regularized return. This is the updated SAC formulation with no separate value network.

$$J(\pi)=\mathbb E\!\left[\sum_{t\ge0}\gamma^t
(r_{t+1}+\alpha\mathcal H(\pi(\cdot\mid s_t)))\right],\qquad
L_\pi=\mathbb E[\alpha\log\pi_\theta(a\mid s)-\min_i Q_{\phi_i}(s,a)].$$

Here $s,a,r,s'$ denote an observation, action, reward and next observation;
$d$ is one only for true termination; $\gamma$ is the discount factor.
$\theta$ and $\phi_i$ are actor and critic weights; bars denote target weights.
$\mu$ is a deterministic policy, $\pi$ a stochastic policy, and expectations
are minibatch averages from replay. $\alpha$ is the entropy temperature,
$\mathcal H(\pi)=-\mathbb E_a\log\pi(a\mid s)$ is entropy, and $t$ indexes time.
TD3's $\epsilon$ is clipped Gaussian target noise, distinct from behavior noise.
Each implementation below uses only Gymnasium, NumPy and PyTorch.

## 1. Set up the experiment

Use one CPU thread and explicit seeds. The environment determines observation and action dimensions. Hyperparameters are named constants; the default 20,000 steps illustrate learning, not a guaranteed convergence threshold.

In [ ]:
import copy
import math

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

SEED = 7
TOTAL_TIMESTEPS = 20_000
BUFFER_SIZE = 50_000
BATCH_SIZE = 128
LEARNING_STARTS = 1_000
HIDDEN_SIZE = 128
GAMMA = 0.99
TAU = 0.005
ACTOR_LR = 3e-4
CRITIC_LR = 3e-4
MAX_GRAD_NORM = 10.0
EVALUATION_EPISODES = 3
RENDER_MODE = "human"
torch.set_num_threads(1)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

ENV_ID = "Pendulum-v1"
probe_env = gym.make(ENV_ID)
try:
    OBS_DIM = int(np.prod(probe_env.observation_space.shape))
    ACTION_SHAPE = probe_env.action_space.shape
    ACTION_DIM = int(np.prod(ACTION_SHAPE))
    LOW = torch.tensor(probe_env.action_space.low.reshape(-1))
    HIGH = torch.tensor(probe_env.action_space.high.reshape(-1))
finally:
    probe_env.close()
SCALE = (HIGH - LOW) / 2
BIAS = (HIGH + LOW) / 2

INITIAL_ALPHA = 0.2
ALPHA_LR = 3e-4
TARGET_ENTROPY = -float(ACTION_DIM) + SCALE.log().sum().item()

## 2. Reparameterize a bounded stochastic policy

$$u=m_\theta(s)+\sigma_\theta(s)\epsilon,\quad \epsilon\sim\mathcal N(0,I),
\quad a=b+c\tanh u,$$
$$\log\pi(a\mid s)=\sum_j[\log\mathcal N(u_j;m_j,\sigma_j)
-\log c_j-\log(1-\tanh^2u_j)].$$

$m$ and $\sigma$ are state-dependent Gaussian means and standard deviations;
$j$ indexes action dimensions and $I$ is the identity covariance.
$b=(h+l)/2$ and $c=(h-l)/2$ rescale to action bounds.
`rsample` preserves the gradient from the critic through the action into the
actor. The log-Jacobian uses a softplus identity for stability when tanh
saturates. `SCALE.log()` includes the change of action units; the entropy target
adds the same log-scale shift. Evaluation squashes the Gaussian mean.
Each critic concatenates the observation and environment action.

In [ ]:
def mlp(input_dim, output_dim):
    return nn.Sequential(
        nn.Linear(input_dim, HIDDEN_SIZE),
        nn.ReLU(),
        nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE),
        nn.ReLU(),
        nn.Linear(HIDDEN_SIZE, output_dim),
    )


class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = mlp(OBS_DIM + ACTION_DIM, 1)

    def forward(self, observations, actions):
        inputs = torch.cat((observations, actions), dim=-1)
        return self.network(inputs)


class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = mlp(OBS_DIM, 2 * ACTION_DIM)

    def forward(self, observations):
        means, log_stds = self.network(observations).chunk(2, dim=-1)
        return means, log_stds.clamp(-20, 2)


actor = Actor()


def sample_policy(observations):
    means, log_stds = actor(observations)
    distribution = torch.distributions.Normal(means, log_stds.exp())
    raw = distribution.rsample()
    actions = BIAS + SCALE * raw.tanh()
    log_jacobian = 2 * (math.log(2) - raw - nn.functional.softplus(-2 * raw))
    log_prob = distribution.log_prob(raw) - log_jacobian - SCALE.log()
    return actions, log_prob.sum(dim=-1, keepdim=True)


@torch.no_grad()
def select_action(observation, deterministic=False):
    state = torch.tensor(observation, dtype=torch.float32).reshape(1, OBS_DIM)
    if deterministic:
        means, _ = actor(state)
        action = BIAS + SCALE * means.tanh()
    else:
        action, _ = sample_policy(state)
    return action.numpy().astype(np.float32).reshape(ACTION_SHAPE)


critics = nn.ModuleList([Critic() for _ in range(2)])
target_critics = copy.deepcopy(critics).requires_grad_(False)
actor_optimizer = torch.optim.Adam(actor.parameters(), lr=ACTOR_LR)
critic_optimizer = torch.optim.Adam(critics.parameters(), lr=CRITIC_LR)
log_alpha = torch.tensor(math.log(INITIAL_ALPHA), requires_grad=True)
alpha_optimizer = torch.optim.Adam([log_alpha], lr=ALPHA_LR)

## 3. Store experience and sample replay

$$B=\{(s,a,r,s',d,\text{truncated})\}_1^N\sim\mathcal D.$$

$\mathcal D$ is a circular experience buffer and $N$ is `BATCH_SIZE`.
Uniformly sampled transitions decorrelate consecutive environment steps.
Store the actual behavior action, not the current actor's action. Keep true
termination separate from time-limit truncation: both reset the environment,
but only termination will suppress the bootstrap in the next cell.

In [ ]:
class ReplayBuffer:
    def __init__(self):
        self.observations = np.empty((BUFFER_SIZE, OBS_DIM), dtype=np.float32)
        self.next_observations = np.empty_like(self.observations)
        self.actions = np.empty((BUFFER_SIZE, ACTION_DIM), dtype=np.float32)
        self.rewards = np.empty((BUFFER_SIZE, 1), dtype=np.float32)
        self.terminated = np.empty((BUFFER_SIZE, 1), dtype=np.float32)
        self.truncated = np.empty_like(self.terminated)
        self.position = 0
        self.size = 0

    def add(self, observation, action, reward, next_observation, terminated, truncated):
        i = self.position
        self.observations[i] = np.asarray(observation).reshape(-1)
        self.actions[i] = np.asarray(action).reshape(-1)
        self.rewards[i] = reward
        self.next_observations[i] = np.asarray(next_observation).reshape(-1)
        self.terminated[i] = terminated
        self.truncated[i] = truncated
        self.position = (i + 1) % BUFFER_SIZE
        self.size = min(self.size + 1, BUFFER_SIZE)

    def sample(self):
        indices = rng.integers(self.size, size=BATCH_SIZE)
        arrays = (
            self.observations,
            self.actions,
            self.rewards,
            self.next_observations,
            self.terminated,
            self.truncated,
        )
        return tuple(torch.tensor(array[indices]) for array in arrays)


replay = ReplayBuffer()

## 4. Form detached Bellman targets

$$y=r+\gamma(1-d)V_{\mathrm{next}}(s'),\qquad
\bar w\leftarrow(1-\tau)\bar w+\tau w.$$

`terminated` implements $d$; `truncated` is deliberately absent from this mask.
`no_grad` keeps target computation outside the optimization graph.
$w$ denotes online network weights and $\bar w$ their slowly moving target
copies; `TAU` is $\tau$. The soft-update helper only moves network parameters.
$V_{\mathrm{next}}=\min_i Q_{\bar\phi_i}(s\prime,a\prime)-\alpha\log\pi_\theta(a\prime\mid s\prime)$, with $a\prime$ sampled from the current actor. SAC needs target critics but no target actor.

In [ ]:
@torch.no_grad()
def soft_update(online, target):
    for parameter, delayed in zip(
        online.parameters(), target.parameters(), strict=True
    ):
        delayed.lerp_(parameter, TAU)


@torch.no_grad()
def td_target(rewards, next_states, terminated):
    actions, log_prob = sample_policy(next_states)
    q = torch.minimum(
        target_critics[0](next_states, actions), target_critics[1](next_states, actions)
    )
    value = q - log_alpha.exp() * log_prob
    return rewards + GAMMA * (1 - terminated) * value

## 5. Fit the critic and improve the actor

$$L_Q=\sum_i\mathbb E_B[(Q_{\phi_i}(s,a)-y)^2].$$

The critic fits detached targets at recorded behavior actions. The actor then
optimizes its current actions at the sampled states. These are different
uses of the same replay minibatch. Gradient clipping bounds each optimizer's
gradient norm; it does not change the Bellman target.

$$L_\pi=\mathbb E[\alpha\log\pi_\theta(a\mid s)-\min_iQ_{\phi_i}(s,a)],\qquad
L_\beta=-\mathbb E[\beta(\log\pi_\theta(a\mid s)+\mathcal H_*)],\quad \alpha=e^\beta.$$

$\beta$ is `log_alpha` and $\mathcal H_*$ is `TARGET_ENTROPY`.
The log-temperature surrogate increases $\alpha$ when entropy is below target.
Detach the entropy residual so the temperature optimizer cannot change the
actor; detach $\alpha$ in actor and critic updates. The continuous version
estimates action expectations with a reparameterized sample; discrete SAC
sums them exactly. Target critics move after every update.

In [ ]:
def update(batch, update_number):
    states, actions, rewards, next_states, terminated, truncated = batch
    targets = td_target(rewards, next_states, terminated)
    critic_loss = sum(
        nn.functional.mse_loss(critic(states, actions), targets) for critic in critics
    )

    critic_optimizer.zero_grad(set_to_none=True)
    critic_loss.backward()
    nn.utils.clip_grad_norm_(critics.parameters(), MAX_GRAD_NORM)
    critic_optimizer.step()
    critic_optimizer.zero_grad(set_to_none=True)
    metrics = {"critic_loss": critic_loss.item()}
    critics.requires_grad_(False)
    sampled_actions, log_prob = sample_policy(states)
    q = torch.minimum(
        critics[0](states, sampled_actions), critics[1](states, sampled_actions)
    )
    actor_loss = (log_alpha.detach().exp() * log_prob - q).mean()

    actor_optimizer.zero_grad(set_to_none=True)
    actor_loss.backward()
    nn.utils.clip_grad_norm_(actor.parameters(), MAX_GRAD_NORM)
    actor_optimizer.step()
    critics.requires_grad_(True)
    temperature_loss = -(log_alpha * (log_prob.detach() + TARGET_ENTROPY)).mean()
    alpha_optimizer.zero_grad(set_to_none=True)
    temperature_loss.backward()
    alpha_optimizer.step()
    metrics.update(
        entropy=-log_prob.detach().mean().item(), alpha=log_alpha.detach().exp().item()
    )
    soft_update(critics, target_critics)
    metrics["actor_loss"] = actor_loss.item()
    return metrics

## 6. Interact and learn

During warmup sample uniformly from the action space; afterward use the
exploratory policy. Each new transition enters replay, then one minibatch
update runs once enough experience exists. Reset on `terminated or truncated`
after storing the final observation. The training objective is a discounted
return, while the plotted episode return $R=\sum_t r_{t+1}$ is undiscounted.
Training owns its environment and closes it even if interrupted.

In [ ]:
def train(total_timesteps):
    env = gym.make(ENV_ID)
    episode_returns, critic_losses, actor_losses = [], [], []
    entropies, temperatures = [], []
    episode_return = 0.0
    updates = 0
    try:
        env.action_space.seed(SEED)
        observation, _ = env.reset(seed=SEED)
        for step in range(total_timesteps):
            action = (
                env.action_space.sample()
                if step < LEARNING_STARTS
                else select_action(observation)
            )
            next_observation, reward, terminated, truncated, _ = env.step(action)
            # Store the final observation before resetting at a time limit.
            replay.add(
                observation, action, reward, next_observation, terminated, truncated
            )
            observation = next_observation
            episode_return += float(reward)
            if replay.size >= BATCH_SIZE and step + 1 >= LEARNING_STARTS:
                updates += 1
                metrics = update(replay.sample(), updates)
                critic_losses.append(metrics["critic_loss"])
                if "actor_loss" in metrics:
                    actor_losses.append(metrics["actor_loss"])
                if "entropy" in metrics:
                    entropies.append(metrics["entropy"])
                    temperatures.append(metrics["alpha"])
            if terminated or truncated:
                episode_returns.append(episode_return)
                episode_return = 0.0
                observation, _ = env.reset()
            if (step + 1) % 5_000 == 0:
                recent = np.mean(episode_returns[-10:]) if episode_returns else np.nan
                print(f"Step {step + 1}: recent return={recent:.1f}")
    finally:
        env.close()
    return episode_returns, critic_losses, actor_losses, entropies, temperatures


returns, critic_losses, actor_losses, entropies, temperatures = train(TOTAL_TIMESTEPS)

## 7. Inspect the learning signals

Plot episode returns and their moving mean
$\bar R_k=\frac{1}{W}\sum_{j=k-W+1}^kR_j$, where $W$ is the window size.
Critic loss measures value fitting; actor loss is an optimization objective,
not a return estimate. For SAC also compare entropy with its target and track
$\alpha$. Short runs and different seeds may behave differently.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(returns, alpha=0.3, label="Episode return")
if returns:
    window = min(10, len(returns))
    average = np.convolve(returns, np.ones(window) / window, mode="valid")
    axes[0].plot(np.arange(window - 1, len(returns)), average, label="Moving mean")
axes[0].set(xlabel="Episode", ylabel="Return", title=ENV_ID)
axes[0].legend()
axes[1].plot(critic_losses, label="Critic")
axes[1].set(xlabel="Critic update", ylabel="MSE", title="Value fitting")
axes[2].plot(actor_losses, label="Actor loss")
axes[2].set(xlabel="Actor update", ylabel="Loss", title="Policy optimization")
plt.tight_layout()
plt.show()
if entropies:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(entropies)
    axes[0].axhline(TARGET_ENTROPY, color="black", linestyle="--")
    axes[0].set(xlabel="Update", ylabel="Entropy", title="Entropy and target")
    axes[1].plot(temperatures)
    axes[1].set(xlabel="Update", ylabel="Alpha", title="Learned temperature")
    plt.tight_layout()
    plt.show()

## 8. Evaluate deterministically in a rendered environment

Use a separate environment and new seeds. Deterministic evaluation removes behavior noise or uses the stochastic policy mode; it measures reward without the entropy bonus. The default `human` mode opens a window; set `RENDER_MODE = "rgb_array"` for headless execution.

In [ ]:
evaluation_env = gym.make(ENV_ID, render_mode=RENDER_MODE)
evaluation_returns = []
try:
    for episode in range(EVALUATION_EPISODES):
        observation, _ = evaluation_env.reset(seed=1_000 + episode)
        total_reward = 0.0
        done = False
        while not done:
            action = select_action(observation, deterministic=True)
            observation, reward, terminated, truncated, _ = evaluation_env.step(action)
            total_reward += float(reward)
            done = terminated or truncated
        evaluation_returns.append(total_reward)
finally:
    evaluation_env.close()
print("Evaluation returns:", evaluation_returns)
print(f"Mean deterministic return: {np.mean(evaluation_returns):.1f}")

## What changed, and what comes next?

Reparameterization allows an off-policy stochastic actor to differentiate through Q. Next, discrete SAC replaces Monte Carlo action expectations with finite sums.

[Original reference](https://arxiv.org/abs/1812.05905) · [Library guide](../../docs/algorithms/sac.md) · [Public API example](../../examples/sac.ipynb)